In [ ]:
import pandas as pd
import re
from data_gatherer.data_gatherer import DataGatherer
from data_gatherer.parser.xml_parser import XMLParser

In [ ]:
# df = pd.read_parquet("scripts/exp_input/Local_fetched_data.parquet")

dg = DataGatherer(log_level='INFO')

if dg.parser is None:
    dg.parser = XMLParser(dg.open_data_repos_ontology, dg.logger, llm_name=dg.llm)

In [ ]:
# there are some pcm ids in this csv article_ids_REV_test.csv that we want to filter the df on
article_ids = pd.read_csv("scripts/exp_input/REV_test.txt", header=None, names=['publication'])['publication'].tolist()
article_ids = [re.sub(r'https://www.ncbi.nlm.nih.gov/pmc/articles/', '', id.lower()) for id in article_ids]
len(article_ids), len(df)

In [ ]:
df_filtered = df[df['publication'].str.lower().isin([id.lower() for id in article_ids])]
df_filtered['format'].value_counts()

In [ ]:
# # Flan-t5-finetuned -- base
# ! bash k8s/run_loop.sh \
# --iterations 1 \
# --gpus 1 \
# --input article_ids_REV_test.csv \
# --max-articles-per-slice 249 \
# --output-dir k8s/output/rev_test_c1 \
# --seed-ontology data_gatherer/config/open_bio_data_repos.json \
# --job-suffix -tc1 \
# --semantic-retrieval false \
# --brute-force-regex false \
# --no-enrich

In [ ]:
# # Flan-t5-finetuned -- S3
# ! bash k8s/run_loop.sh \
#     --iterations 1 --gpus 1 \
#     --input article_ids_REV_test.csv \
#     --max-articles-per-slice 249 \
#     --output-dir k8s/output/rev_test_c2 \
#     --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#     --job-suffix -tc2 \
#     --semantic-retrieval true \
#     --top-k 3 \
#     --brute-force-regex false \
#     --no-enrich

In [ ]:
# # Flan-t5-finetuned -- RS3
# ! bash k8s/run_loop.sh \
#     --iterations 1 --gpus 1 \
#     --input article_ids_REV_test.csv \
#     --max-articles-per-slice 249 \
#     --output-dir k8s/output/rev_test_c3 \
#     --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#     --job-suffix -tc3 \
#     --semantic-retrieval true \
#     --top-k 3 \
#     --brute-force-regex true \
#     --no-enrich

In [ ]:
# Flan-t5-finetuned -- R
! bash k8s/run_loop.sh \
    --iterations 1 --gpus 1 \
    --input article_ids_REV_test.csv \
    --max-articles-per-slice 249 \
    --output-dir k8s/output/rev_test_c5 \
    --seed-ontology data_gatherer/config/open_bio_data_repos.json \
    --job-suffix -tc5 \
    --semantic-retrieval false \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# # Flan-t5-finetuned -- Full Document Chunk
# ! bash k8s/run_loop.sh \
#       --iterations 1 --gpus 1 \
#       --input article_ids_REV_test.csv \
#       --max-articles-per-slice 249 \
#       --output-dir k8s/output/rev_test_c4 \
#       --seed-ontology data_gatherer/config/open_bio_data_repos.json \
#       --job-suffix -tc4 \
#       --top-k all \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --no-enrich

In [ ]:
# from scripts.experiment_utils import compute_gpu_energy_wh

# compute_gpu_energy_wh("k8s/output/rev_test_c1/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c2/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c3/iter1/slice_1_gpu_power.csv")

In [ ]:
# compute_gpu_energy_wh("k8s/output/rev_test_c4/iter1/slice_1_gpu_power.csv")

In [ ]:
# # Claude Haiku 4.5 -- base 
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c1 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c2 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 4.5 -- RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c3 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true \
#     --use-batch-api false

In [ ]:
# Claude Haiku 4.5 -- R
! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
    --output-dir k8s/output/rev_test_haiku_c5 \
    --model claude-haiku-4-5-20251001 --batch-size 249 \
    --brute-force-regex true --semantic-retrieval false 

In [ ]:
# # Claude Haiku 4.5 — FDR
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_haiku_c4 \
#     --model claude-haiku-4-5-20251001 --batch-size 249 \
#     --full-document-read true \
#     --prompt-name CLAUDE_FDR_FewShot

In [ ]:
batch_id = 'msgbatch_01KCiZ4gBaXBMvidxusWBnQg'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='anthropic',
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_haiku_c5/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - Base
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c1b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false 

In [ ]:
batch_id = 'batch_6a51ec4742a881909cd044d78b802ca1'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c1/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - S3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c2b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex false

In [ ]:
batch_id = 'batch_6a51ececf1e481908f3e5330e7918ccb'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c2/dataset_citations.csv')

In [ ]:
# # gpt 5 mini - RS3
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c3b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval true --brute-force-regex true


In [ ]:
# gpt 5 mini - R
! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
    --output-dir k8s/output/rev_test_gpt5mini_c5 \
    --model gpt-5-mini --batch-size 249 \
    --semantic-retrieval false --brute-force-regex true

In [ ]:
batch_id = 'batch_6a58a103e25881909d2587617ac3cfbe'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c5/dataset_citations.csv')

In [ ]:
# # gpt-5-mini — FDR
# ! python k8s/k8s_processor.py --input k8s/input/article_ids_REV_test.csv \
#     --output-dir k8s/output/rev_test_gpt5mini_c4b \
#     --model gpt-5-mini --batch-size 249 \
#     --semantic-retrieval false --brute-force-regex false \
#     --full-document-read true \
#     --prompt-name GPT_FDR_FewShot

In [ ]:
batch_id = 'batch_6a51ee2920ac8190a58188de3b2db4b2'

res = dg.parser.llm_client.download_batch_results(
    batch_id=batch_id,
    output_file_path='scripts/tmp/resp_RTR_base.jsonl',
    api_provider='openai'
)

res_df = dg.from_batch_resp_file_to_df('scripts/tmp/resp_RTR_base.jsonl', output_file_path='k8s/output/rev_test_gpt5mini_c4/dataset_citations.csv')

In [ ]:
# # Gemini 3.5 -- base
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c1 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval false \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- S3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c2 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex false \
#       --use-batch-api false

In [ ]:
# # Gemini 3.5 -- RS3
# !python k8s/k8s_processor.py \
#       --input k8s/input/article_ids_REV_test.csv \
#       --output-dir k8s/output/rev_test_gemini_c3 \
#       --model gemini-3.5-flash \
#       --batch-size 249 \
#       --semantic-retrieval true \
#       --brute-force-regex true \
#       --use-batch-api false

In [ ]:
# Gemini 3.5 -- R
!python k8s/k8s_processor.py \
      --input k8s/input/article_ids_REV_test.csv \
      --output-dir k8s/output/rev_test_gemini_c5 \
      --model gemini-3.5-flash \
      --batch-size 249 \
      --semantic-retrieval false \
      --brute-force-regex true \
      --use-batch-api false

In [ ]:
# # Gemini 3.5 -- FDR
# !python k8s/k8s_processor.py \
# --input k8s/input/article_ids_REV_test.csv \
# --output-dir k8s/output/rev_test_gemini_c4 \
# --model gemini-3.5-flash \
# --batch-size 249 \
# --full-document-read true \
# --use-batch-api false \
# --prompt-name GPT_FDR_FewShot


In [ ]:
article_ids_synapse = [re.sub(r'pmc:', '', pmcid) for pmcid in pd.read_csv("scripts/exp_input/syn66046424-20260629.csv")["pmcid"].tolist()]
article_ids_synapse

## Results

In [ ]:
! python scripts/BioDMS/eval_configs.py

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results = pd.read_csv("scripts/BioDMS/config_eval/results.csv")
results["model"] = results["config"].str.split(" · ").str[0]
results["variant"] = results["config"].str.extract(r"(c\d)")

models = ["T5", "Haiku", "GPT-5-mini", "Gemini-3.5-flash"]
variants = ["c1", "c2", "c3", "c4"]
variant_labels = {
    "c1": "base",
    "c2": "S3",
    "c3": "RS3",
    "c4": "FDR",
}
# model brand colors (verified against each company's actual brand assets):
#   Claude/Anthropic terracotta #D97757, OpenAI/ChatGPT teal #10A37F, Google Gemini blue #078EFA.
#   T5/Flan-T5 has no official brand color (research model, not a branded product) -> neutral gray.
model_colors = {
    "T5": "#6b7280",
    "Haiku": "#d97757",
    "GPT-5-mini": "#10a37f",
    "Gemini-3.5-flash": "#078efa",
}
metric = "gold_R"

# legend labels shortened so a single-row legend fits the column width
legend_labels = {
    "T5": "T5",
    "Haiku": "Haiku",
    "GPT-5-mini": "GPT-5-mini",
    "Gemini-3.5-flash": "Gemini-3.5",
}

# figsize matches the printed single-column width (~3.4in in ACM two-column layout) so
# fonts set here map ~1:1 to printed point sizes instead of being shrunk 3x on include
fig, ax = plt.subplots(figsize=(3.4, 2.5), facecolor="#fcfcfb", dpi=300)
ax.set_facecolor("#fcfcfb")

x = np.arange(len(variants))
width = 0.2

for i, model in enumerate(models):
    sub = results[results["model"] == model].set_index("variant")
    vals = [sub.loc[v, metric] if v in sub.index else np.nan for v in variants]
    offset = (i - (len(models) - 1) / 2) * width
    bars = ax.bar(x + offset, vals, width, label=legend_labels[model], color=model_colors[model],
                   edgecolor="#fcfcfb", linewidth=0.6)
    # rotated 90 deg so labels stay narrow and don't collide between adjacent bars
    ax.bar_label(bars, fmt="%.2f", padding=1, fontsize=3.5, color="#52514e")

ax.set_xticks(x)
ax.set_xticklabels([variant_labels[v] for v in variants], fontsize=9, color="#222")
ax.set_ylim(0, 1.1)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.tick_params(axis="y", labelsize=4, colors="#52514e")
ax.tick_params(axis="x", length=0)

# single-row legend above the axes: reclaims the horizontal space a side legend would
# take from the bars, at the cost of a bit of vertical space (freed up by dropping the title)
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False,
    fontsize=6.5, handlelength=1.0, handletextpad=0.4, columnspacing=0.9, borderaxespad=0.1,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.spines["bottom"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("scripts/BioDMS/config_eval/gold_recall_chart.png", dpi=300,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
